In [1]:
import numpy as np
import open3d as o3d
from plyfile import PlyData 


/Users/adeleyounis/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Step 1. Read both meshes in and convert to point clouds.

pixie_mesh = Mesh from PIXIE model
alpha_mesh = Mesh derived geometrically from depth data
 

In [2]:
# read in the meshes
alpha_mesh = o3d.io.read_triangle_mesh("adele_up_mesh.obj")
alpha_mesh.compute_vertex_normals()

pixie_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/volume-model/output/bulky/bulky.obj")
pixie_mesh.compute_vertex_normals()

alpha_mesh.paint_uniform_color([0.7, 0.7, 0.7])
# pixie_mesh.paint_uniform_color([1.0, 0.2, 0.2])

# verify that they are triangular meshes with points and triangles
print(f"alpha_mesh: {alpha_mesh}")
print(f"pixie_mesh: {pixie_mesh}")

o3d.visualization.draw_geometries(
    [pixie_mesh],
    mesh_show_back_face=True
)

alpha_mesh: TriangleMesh with 7726 points and 15910 triangles.
pixie_mesh: TriangleMesh with 11313 points and 20908 triangles.


In [3]:
# scale and orient since pixie is scaled down
depth_size = np.array(alpha_mesh.get_max_bound() - alpha_mesh.get_min_bound())
pixie_size = np.array(pixie_mesh.get_max_bound() - pixie_mesh.get_min_bound())

scale =  pixie_size.max() / depth_size.max()
print(scale)

alpha_mesh.scale(scale, center=(0,0,0))

# Flip 180° around X (fix upside down)
pixie_mesh.rotate(pixie_mesh.get_rotation_matrix_from_xyz((np.pi, 0, 0)), center=(0,0,0))

# Rotate 180° around Y (face the camera)
pixie_mesh.rotate(pixie_mesh.get_rotation_matrix_from_xyz((0, np.pi, 0)), center=(0,0,0))

depth_center = alpha_mesh.get_center()
pixie_center = pixie_mesh.get_center()

pixie_mesh.translate(depth_center - pixie_center)

0.31864262385616277


TriangleMesh with 11313 points and 20908 triangles.

In [4]:
# get head from pixie model - delete alpha model head
min_bound = alpha_mesh.get_min_bound()
max_bound = alpha_mesh.get_max_bound()

print(min_bound, max_bound)

verts = np.asarray(alpha_mesh.vertices)
y_min, y_max = min_bound[1], max_bound[1]
x_min, x_max = min_bound[0], max_bound[0]
x_mean = (x_min + x_max) / 2

# Depth arms = points far left or far right
# (tighter threshold = fewer points kept)
arm_threshold = 0.25 * (x_max - x_min)

left_arm_mask  = np.asarray(alpha_mesh.vertices)[:,0] < (x_mean - arm_threshold)
right_arm_mask = np.asarray(alpha_mesh.vertices)[:,0] > (x_mean + arm_threshold)

depth_arm_mask = left_arm_mask | right_arm_mask
head_mask = verts[:,1] > (y_min + 0.820*(y_max - y_min))  # top 20%


[-0.55782534 -0.74071664  2.4172866 ] [0.6120647  0.87482059 3.17079367]


In [5]:
# convert to point cloud
alpha_pcd = alpha_mesh.sample_points_uniformly(number_of_points=50000)
pixie_pcd = pixie_mesh.sample_points_uniformly(number_of_points=50000)

alpha_pcd.estimate_normals()
pixie_pcd.estimate_normals()

print(len(alpha_pcd.points))
print(len(pixie_pcd.points))

50000
50000


Step 1. Rigid Transformation:
Aligns PIXIE to depth mesh using ICP (point-to-plane)


In [6]:
reg = o3d.pipelines.registration.registration_icp(
    source=pixie_pcd,
    target=alpha_pcd,
    max_correspondence_distance=0.5,
    init=np.eye(4),
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(
        max_iteration=5000)
)

pixie_mesh.transform(reg.transformation)



TriangleMesh with 11313 points and 20908 triangles.

In [16]:
o3d.visualization.draw_geometries(
    [alpha_mesh, pixie_mesh],
    mesh_show_back_face=True
)

Step 2: Identify PIXIE mesh parts to remove


In [ ]:
pixie_normals = np.asarray(pixie_mesh.vertex_normals)
pixie_vertices_full = np.asarray(pixie_mesh.vertices)

head_mask_pixie = pixie_vertices_full[:,1] > (
    pixie_vertices_full[:,1].min() + 0.70 * (pixie_vertices_full[:,1].ptp())
)

# Arms = x far from center
x_min, x_max = pixie_vertices_full[:,0].min(), pixie_vertices_full[:,0].max()
x_center = (x_min + x_max) / 2
arm_thresh = 0.25 * (x_max - x_min)

arm_mask_pixie = (pixie_vertices_full[:,0] < x_center - arm_thresh) | \
                 (pixie_vertices_full[:,0] > x_center + arm_thresh)

# Keep head + arms from PIXIE
pixie_head_arms_mask = head_mask_pixie | arm_mask_pixie
pixie_head_arms = pixie_mesh.select_by_index(
    np.where(pixie_head_arms_mask)[0].tolist(),
    cleanup=True
)

pixie_normals = np.asarray(pixie_mesh.vertex_normals)
back_mask = pixie_normals[:,2] >= -0.25   # normals NOT facing camera

pixie_back_mask = back_mask & (~pixie_head_arms_mask)
pixie_back_only = pixie_mesh.select_by_index(
    np.where(pixie_back_mask)[0].tolist(),
    cleanup=True
)

depth_verts = np.asarray(alpha_mesh.vertices)

head_mask_depth = depth_verts[:,1] > (
    depth_verts[:,1].min() + 0.80 * depth_verts[:,1].ptp()
)

x_min, x_max = depth_verts[:,0].min(), depth_verts[:,0].max()
x_center = (x_min + x_max) / 2
arm_thresh = 0.25 * (x_max - x_min)
arm_mask_depth = (depth_verts[:,0] < x_center - arm_thresh) | \
                 (depth_verts[:,0] > x_center + arm_thresh)

remove_depth_mask = head_mask_depth | arm_mask_depth
keep_depth_mask = ~remove_depth_mask

depth_filtered = alpha_mesh.select_by_index(
    np.where(keep_depth_mask)[0].tolist(),
    cleanup=True
)

alpha_pcd = depth_filtered.sample_points_poisson_disk(50000)
tree = o3d.geometry.KDTreeFlann(alpha_pcd)

pixie_back_vertices = np.asarray(pixie_back_only.vertices)
keep = []

for i, v in enumerate(pixie_back_vertices):
    _, idx, dist = tree.search_knn_vector_3d(v, 1)
    if dist[0] > 0.0001:
        keep.append(i)

pixie_back_pruned = pixie_back_only.select_by_index(keep, cleanup=True)

combined = pixie_head_arms + pixie_back_pruned + depth_filtered
combined.remove_duplicated_vertices()
combined.remove_duplicated_triangles()
combined.remove_non_manifold_edges()
combined.compute_vertex_normals()

# alpha_pcd = alpha_mesh.sample_points_poisson_disk(50000)
# alpha_tree = o3d.geometry.KDTreeFlann(alpha_pcd)

# pixie_vertices = np.asarray(pixie_back_only.vertices)
# head_mask = pixie_vertices[:,1] > (pixie_vertices[:,1].mean() + 0.15)

# keep = []

# for i, v in enumerate(pixie_vertices):
#     _, idx, dist = alpha_tree.search_knn_vector_3d(v, 1)
#     if dist[0] > 0.0001:   
#         keep.append(i)

# pixie_clean = pixie_back_only.select_by_index(keep)

# combined = pixie_clean + depth_filtered

# combined = combined.filter_smooth_taubin(number_of_iterations=10)
# combined.compute_vertex_normals()

# combined.remove_duplicated_vertices()
# combined.remove_duplicated_triangles()
# combined.remove_non_manifold_edges()
# combined.compute_vertex_normals()

o3d.visualization.draw_geometries(
    [combined],
    mesh_show_back_face=True
)

# pcd = combined.sample_points_poisson_disk(80000)
# o3d.io.write_point_cloud("combined_point_cloud.ply", pcd)


[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


: 